# Dataset Scraping Mentah

In [2]:
import pandas as pd
import numpy as np
import os
import re

RAW_DIR = "../data/raw" if os.path.exists("../data/raw") else "data/raw"

print("=" * 65)
print("PENGGABUNGAN & STANDARISASI DATASET POLITIK INDONESIA")
print("=" * 65)

# 1. Aturan Sub-Topik Politik Otomatis (Untuk Analisis Bab 4)
SUB_TOPIC_RULES = {
    "pemilu": ["pemilu", "pilpres", "pilkada", "kpu", "bawaslu", "surat suara", "sirekap", "coblos", "capres", "cawapres", "debat"],
    "kebijakan": ["bansos", "subsidi", "bbm", "ikn", "apbn", "pajak", "omnibus", "undang-undang", "perpres", "aturan", "tarif"],
    "tokoh": ["jokowi", "prabowo", "gibran", "anies", "ganjar", "megawati", "menteri", "presiden", "pejabat", "mahfud"],
    "hukum": ["korupsi", "kpk", "mk", "mahkamah", "polri", "kejaksaan", "tangkap", "ott", "sidang", "vonis", "hakim"]
}

def get_sub_topic(text):
    text_lower = str(text).lower()
    for topic, kws in SUB_TOPIC_RULES.items():
        if any(kw in text_lower for kw in kws):
            return topic
    return "umum"

# ====================================================================
# A. MEMPROSES DATA HOAKS (TurnBackHoax)
# ====================================================================
print("\n--> [1/3] Memproses data Hoaks dari TurnBackHoax...")
df_tbh = pd.read_excel(os.path.join(RAW_DIR, "dataset_turnbackhoax_10_cleaned.xlsx"))

hoax_records = []
for _, row in df_tbh.iterrows():
    # Ambil judul dan buang tag [SALAH] / [DISINFORMASI]
    raw_title = str(row.get('Title', ''))
    clean_title = re.sub(r'\[(SALAH|DISINFORMASI|MANIPULASI|MISLEADING|FITNAH)\]', '', raw_title, flags=re.IGNORECASE).strip()
    
    # Ambil narasi murni (utamakan Clean Narasi)
    narasi = str(row.get('Clean Narasi', row.get('Narasi', ''))).strip()
    
    # Gabungkan judul + narasi
    full_text = f"{clean_title}. {narasi}".strip()
    full_text = re.sub(r'\s+', ' ', full_text)
    
    # Filter Politik: Cek kolom politik bawaan atau pencocokan kata kunci
    sub_topic = get_sub_topic(full_text)
    is_politik = (row.get('politik') == 1) or (sub_topic != "umum")
    
    if len(full_text) >= 30 and is_politik:
        hoax_records.append({
            "text": full_text[:1500],
            "label": 1, # 1 = HOAKS
            "sub_topic": sub_topic if sub_topic != "umum" else "tokoh",
            "source": "turnbackhoax.id",
            "date": str(row.get('Timestamp', ''))
        })

df_hoax_clean = pd.DataFrame(hoax_records).drop_duplicates(subset=['text'])
print(f"    Berhasil memfilter {len(df_hoax_clean)} hoaks politik steril.")

# ====================================================================
# B. MEMPROSES DATA FAKTA (CNN, Tempo, Kompas)
# ====================================================================
print("\n--> [2/3] Memproses data Fakta dari CNN, Tempo, dan Kompas...")

fact_sources = [
    ("dataset_cnn_10k_cleaned.xlsx", "cnnindonesia.com"),
    ("dataset_tempo_6k_cleaned.xlsx", "tempo.co"),
    ("dataset_kompas_4k_cleaned.xlsx", "kompas.com")
]

fact_records = []
for file_name, source_name in fact_sources:
    file_path = os.path.join(RAW_DIR, file_name)
    if os.path.exists(file_path):
        df_fact_raw = pd.read_excel(file_path)
        print(f"    Membaca {file_name} ({len(df_fact_raw)} baris)...")
        
        for _, row in df_fact_raw.iterrows():
            title = str(row.get('Title', '')).strip()
            # Utamakan text_new, fallback ke FullText
            body = str(row.get('text_new', row.get('FullText', ''))).strip()
            
            full_text = f"{title}. {body}".strip()
            full_text = re.sub(r'\s+', ' ', full_text)
            
            # Saring agar hanya mengambil yang bertema politik
            sub_topic = get_sub_topic(full_text)
            if len(full_text) >= 30 and sub_topic != "umum":
                fact_records.append({
                    "text": full_text[:1500],
                    "label": 0, # 0 = FAKTA
                    "sub_topic": sub_topic,
                    "source": source_name,
                    "date": str(row.get('Timestamp', ''))
                })

df_fact_clean = pd.DataFrame(fact_records).drop_duplicates(subset=['text'])
print(f"    Berhasil memfilter {len(df_fact_clean)} fakta politik terverifikasi.")

# ====================================================================
# C. BALANCING DATASET 50:50 & SIMPAN
# ====================================================================
print("\n--> [3/3] Melakukan penyeimbangan data (Balancing 50:50)...")

target_samples = min(len(df_hoax_clean), len(df_fact_clean))
print(f"    Menyamakan jumlah sampel ke: {target_samples} baris per kelas.")

df_hoax_bal = df_hoax_clean.sample(n=target_samples, random_state=42)
df_fact_bal = df_fact_clean.sample(n=target_samples, random_state=42)

# Gabungkan & Acak (Shuffle)
master_dataset = pd.concat([df_hoax_bal, df_fact_bal]).sample(frac=1, random_state=42).reset_index(drop=True)

# Beri ID unik
master_dataset.insert(0, 'id', [f"POL-{i+1:05d}" for i in range(len(master_dataset))])

# Simpan ke folder raw dan processed
output_path = os.path.join(RAW_DIR, "dataset_politik_balanced.csv")
master_dataset.to_csv(output_path, index=False, encoding='utf-8-sig')

print("\n" + "=" * 65)
print(f"🎉 SUKSES BESAR! Master Dataset Tersimpan di: {output_path}")
print(f"Total Baris Data    : {len(master_dataset):,} baris")
print(f"Distribusi Kelas    : {master_dataset['label'].value_counts().to_dict()} (1=Hoaks, 0=Fakta)")
print("=" * 65)
print("\nSebaran Per Sub-Topik Politik:")
print(master_dataset['sub_topic'].value_counts())
print("\nContoh 3 Baris Pertama:")
display(master_dataset[['id', 'sub_topic', 'label', 'source', 'text']].head(3))

PENGGABUNGAN & STANDARISASI DATASET POLITIK INDONESIA

--> [1/3] Memproses data Hoaks dari TurnBackHoax...
    Berhasil memfilter 4324 hoaks politik steril.

--> [2/3] Memproses data Fakta dari CNN, Tempo, dan Kompas...
    Membaca dataset_cnn_10k_cleaned.xlsx (9630 baris)...
    Membaca dataset_tempo_6k_cleaned.xlsx (6592 baris)...
    Membaca dataset_kompas_4k_cleaned.xlsx (4750 baris)...
    Berhasil memfilter 19106 fakta politik terverifikasi.

--> [3/3] Melakukan penyeimbangan data (Balancing 50:50)...
    Menyamakan jumlah sampel ke: 4324 baris per kelas.

🎉 SUKSES BESAR! Master Dataset Tersimpan di: ../data/raw\dataset_politik_balanced.csv
Total Baris Data    : 8,648 baris
Distribusi Kelas    : {0: 4324, 1: 4324} (1=Hoaks, 0=Fakta)

Sebaran Per Sub-Topik Politik:
sub_topic
tokoh        3373
pemilu       2789
kebijakan    1364
hukum        1122
Name: count, dtype: int64

Contoh 3 Baris Pertama:


,id,sub_topic,label,source,text
0,POL-00001,pemilu,0,tempo.co,"Ditanya Peluang Jadi Cawapres Anies Baswedan, ..."
1,POL-00002,tokoh,1,turnbackhoax.id,Akun Telegram Lembaga Penjamin Simpanan Menawa...
2,POL-00003,hukum,1,turnbackhoax.id,Video “Seorang perwira wercok nyamar jadi maha...


# ScrapingData id_hoax_news

In [ ]:
from dataset import load_dataset
import pandas as pd

print("--> Menarik data hoaks dari Hugging Face (id_hoax_news)...")
ds = load_dataset("id_hoax_news")
df_hf_hoax = ds['train'].to_pandas()

print("Berhasil mengambil data hoaks tambahan:", len(df_hf_hoax))